Git-repo: https://github.com/oggefaderen/CompSci.git
# Contributions
Lovro: ...

Oskar: ...

Uffe: ...

## Part 1: Mixing Patterns and Assortativity

### Setup: Imports and graph construction

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict
import ast
import random
from tqdm import tqdm
import numpy as np

In [ ]:
df = pd.read_csv('./D2_temp_papers.csv')
df['author_ids'] = df['author_ids'].apply(ast.literal_eval)

G = nx.Graph()
pair_counts = defaultdict(int)

for author_list in df['author_ids']:
    for i in range(len(author_list)):
        for j in range(i + 1, len(author_list)):
            pair = tuple(sorted([author_list[i].strip(), author_list[j].strip()]))
            pair_counts[pair] += 1

weighted_edgelist = [(a, b, count) for (a, b), count in pair_counts.items()]
G.add_weighted_edges_from(weighted_edgelist)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

### Load country attributes

In [ ]:
authors_df = pd.read_csv('./final_authors.csv')
country_map = dict(zip(authors_df['id'], authors_df['country_code']))

for node in G.nodes():
    G.nodes[node]['country'] = country_map.get(node, None)

countries_assigned = sum(1 for n in G.nodes() if G.nodes[n]['country'] is not None)
print(f"Country assigned to {countries_assigned}/{G.number_of_nodes()} nodes")

## Q1: Country Assortativity Coefficient

Calculate the Assortativity Coefficient for the network based on the country of each node. Implement the calculation using the formula provided during the lecture (Newman equation 2). Do not use the NetworkX implementation.

In [ ]:
# Q1: Country assortativity using Newman equation 2 (mixing matrix approach)
# e_ij = fraction of edges connecting country i to country j
# r = (Tr(e) - ||e^2||) / (1 - ||e^2||)

edge_type_counts = defaultdict(int)
total_edges = 0

for u, v in G.edges():
    c_u = G.nodes[u].get('country')
    c_v = G.nodes[v].get('country')
    if c_u is None or c_v is None:
        continue
    # Treat as undirected: always store both orderings
    edge_type_counts[(c_u, c_v)] += 1
    if c_u != c_v:
        edge_type_counts[(c_v, c_u)] += 1
    total_edges += 1

countries = sorted(set(c for (c, _) in edge_type_counts))
M = 2 * total_edges  # denominator: counts each edge in both directions

# e_ij fraction
e = {(c_i, c_j): edge_type_counts.get((c_i, c_j), 0) / M for c_i in countries for c_j in countries}

# Trace and marginal sums
trace_e = sum(e[(c, c)] for c in countries)
a = {c: sum(e[(c, c2)] for c2 in countries) for c in countries}
sum_a_sq = sum(a[c]**2 for c in countries)

original_country_r = (trace_e - sum_a_sq) / (1 - sum_a_sq)
print(f"Country assortativity r = {original_country_r:.4f}")

## Q2: Configuration Model

Implement the configuration model using the double edge swap algorithm to generate random networks. Ensure each node retains its original degree but with altered connections.

In [ ]:
def configuration_mode_swap(G):
    G_copy = G.copy()
    edges = list(G_copy.edges())
    E = len(edges)
    num_swaps = E * 10
    swaps_done = 0

    while swaps_done < num_swaps:
        e1, e2 = random.sample(edges, 2)
        u, v = e1
        x, y = e2

        # Flip e1 50% of the time to remove directional bias
        if random.random() < 0.5:
            u, v = v, u

        # Ensure swap nodes are distinct
        if u == y or v == x:
            continue

        # Only swap if new edges don't already exist
        if not G_copy.has_edge(u, y) and not G_copy.has_edge(x, v):
            G_copy.remove_edge(*e1)
            G_copy.remove_edge(*e2)
            G_copy.add_edge(u, y)
            G_copy.add_edge(x, v)

            edges.remove(e1)
            edges.remove(e2)
            edges.append((u, y))
            edges.append((x, v))

            swaps_done += 1

    return G_copy

### Degree sequence verification

Verify the configuration model preserves the degree sequence.

In [ ]:
original_degrees = dict(G.degree())
G_rand = configuration_mode_swap(G)
rand_degrees = dict(G_rand.degree())
assert original_degrees == rand_degrees, "Degrees don't match!"
print("Degree sequence preserved — configuration model is correct")

## Q3: Country Assortativity in Random Networks

Generate 100 random networks using the configuration model. For each, compute country assortativity and plot the distribution. Compare with the original network to assess whether same-country connections are significantly above chance.

In [ ]:
# Q3: Country assortativity distribution over 100 random networks
country_rs_random = []
for i in tqdm(range(100)):
    G_rand = configuration_mode_swap(G)
    # Assign country attributes to the randomized graph nodes
    for node in G_rand.nodes():
        G_rand.nodes[node]['country'] = country_map.get(node, None)
    r_rand = nx.attribute_assortativity_coefficient(G_rand, 'country')
    country_rs_random.append(r_rand)

plt.figure(figsize=(10, 6))
plt.hist(country_rs_random, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
plt.axvline(original_country_r, color='red', linestyle='dashed',
            label=f'Original r = {original_country_r:.4f}')
plt.title('Distribution of Country Assortativity in 100 Random Networks')
plt.xlabel('Country Assortativity Coefficient')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

## Q4: Degree Assortativity

Calculate degree assortativity for the network using the formula discussed in the lecture.

In [ ]:
def degree_assortativity(G):
    edges = list(G.edges())
    ku = np.array([G.degree(u, weight='weight') for u, v in edges])
    kv = np.array([G.degree(v, weight='weight') for u, v in edges])

    numerator = np.mean(ku * kv) - np.mean(ku) * np.mean(kv)
    denominator = np.mean(ku**2) - np.mean(ku)**2
    return numerator / denominator if denominator != 0 else 0

original_degree_r = degree_assortativity(G)
print(f"Degree assortativity coefficient: {original_degree_r:.4f}")

## Q5: Degree Assortativity in Random Networks

Compare the network's degree assortativity against 100 random networks generated via the configuration model. Analyze whether high-degree scientists tend to connect with other high-degree scientists.

In [ ]:
# Q5: Degree assortativity distribution over 100 random networks
degree_rs_random = []
for i in tqdm(range(100)):
    G_rand = configuration_mode_swap(G)
    r_rand = degree_assortativity(G_rand)
    degree_rs_random.append(r_rand)

plt.figure(figsize=(10, 6))
plt.hist(degree_rs_random, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
plt.axvline(original_degree_r, color='red', linestyle='dashed',
            label=f'Original r = {original_degree_r:.4f}')
plt.title('Distribution of Degree Assortativity in 100 Random Networks')
plt.xlabel('Degree Assortativity Coefficient')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()